<a href="https://colab.research.google.com/github/ajaynice1996/DIY_MRI_Workshop_II_Recon/blob/main/Day_3_Notebook_1_2_ZSSR_Low_Field_MRI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1.2 — Zero-Shot Super-Resolution (ZSSR) for Low-Field MRI

## Workshop objective

In this notebook we build a simple, transparent **2D Zero-Shot Super-Resolution (ZSSR)** pipeline and apply it to individual slices of a 64 mT low-field MRI volume.

The learning idea is:

**Father:** a low-resolution (LR) slice  
**Son:** the same LR slice downsampled again  
**CNN:** learns to map son → father  
**Inference:** apply the trained CNN to the original LR slice to estimate a higher-resolution slice.

> **Important:** This notebook is deliberately a teaching implementation. It demonstrates the core ZSSR/self-supervised idea rather than reproducing every detail of the original ZSSR paper.

### What we will compare

1. Bicubic interpolation
2. ZSSR CNN
3. PSNR and SSIM when a reference image is available
4. Visual difference/error maps

For real low-field MRI without a registered high-resolution reference, PSNR/SSIM are not valid as direct quality measures. We therefore keep the 3T image available for **controlled evaluation**, while the ZSSR model itself is trained only from the low-field image.


## Module 1 — Imports and configuration

We keep the implementation small so students can follow every stage of the pipeline.


In [ ]:
import os
import random
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from skimage.transform import resize
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SCALE = 2
PATCH_SIZE = 32
BATCH_SIZE = 32
EPOCHS = 100
LEARNING_RATE = 1e-3

print('Device:', DEVICE)
print('Scale:', SCALE)


## Module 2 — Load the 3T and 64 mT MRI volumes

The paths below follow the workshop structure provided in the previous notebook.

**Teaching point:** The 64 mT image is the image from which ZSSR learns. The 3T image is retained as a possible reference for evaluation when the two volumes are geometrically compatible and registered.


In [ ]:
path_3T = os.path.join(
    workshop_dir,
    '3T',
    'POCEMR001_T1.nii.gz'
)

print('Loading 3T:', path_3T)
nii_3T = nib.load(path_3T)
mri_3T = nii_3T.get_fdata().astype(np.float32)

path_64mT = os.path.join(
    workshop_dir,
    '64mT',
    'POCEMR001_T1.nii.gz'
)

print('Loading 64 mT:', path_64mT)
nii_64mT = nib.load(path_64mT)
mri_64mT = nii_64mT.get_fdata().astype(np.float32)

print('\n' + '=' * 60)
print('3T MRI')
print('=' * 60)
print('Shape:', mri_3T.shape)
print('Voxel spacing:', nii_3T.header.get_zooms()[:3])
print('Data type:', mri_3T.dtype)
print('Minimum intensity:', mri_3T.min())
print('Maximum intensity:', mri_3T.max())
print('Mean intensity:', mri_3T.mean())
print('Standard deviation:', mri_3T.std())

print('\n' + '=' * 60)
print('64 mT Low-Field MRI')
print('=' * 60)
print('Shape:', mri_64mT.shape)
print('Voxel spacing:', nii_64mT.header.get_zooms()[:3])
print('Data type:', mri_64mT.dtype)
print('Minimum intensity:', mri_64mT.min())
print('Maximum intensity:', mri_64mT.max())
print('Mean intensity:', mri_64mT.mean())
print('Standard deviation:', mri_64mT.std())


## Module 3 — Intensity normalization

MRI intensity values are not generally standardized like CT Hounsfield units. For this workshop we use robust percentile normalization so the CNN works with values approximately in `[0, 1]`.

This is a computational normalization step, not a biological calibration.


In [ ]:
def robust_normalize(volume, low=1, high=99):
    lo, hi = np.percentile(volume, [low, high])
    volume = np.clip(volume, lo, hi)
    return ((volume - lo) / (hi - lo + 1e-8)).astype(np.float32)

mri_3T_n = robust_normalize(mri_3T)
mri_64mT_n = robust_normalize(mri_64mT)

print('3T normalized range:', mri_3T_n.min(), mri_3T_n.max())
print('64 mT normalized range:', mri_64mT_n.min(), mri_64mT_n.max())


## Module 4 — Select and inspect one 2D slice

We begin with one slice. This keeps the experiment understandable before we automate it over the complete volume.

Change `SLICE_INDEX` to explore other slices.


In [ ]:
SLICE_INDEX = mri_64mT_n.shape[2] // 2

lr_slice = mri_64mT_n[:, :, SLICE_INDEX]

plt.figure(figsize=(6, 6))
plt.imshow(lr_slice.T, cmap='gray', origin='lower')
plt.title(f'64 mT low-field slice — index {SLICE_INDEX}')
plt.axis('off')
plt.show()


## Module 5 — The ZSSR father–son idea

Suppose the input low-field slice is the **father** image:

`Father = LR`

We create a smaller **son** image by downsampling the father by the same scale:

`Son = downsample(Father, ×2)`

The CNN learns:

`Son → Father`

Then, at inference time:

`Father → predicted high-resolution image`

The original 3T image is **not used as a training target** in this ZSSR experiment.


## Module 6 — Create the father and son images

We downsample the 64 mT slice by ×2 using area averaging. This creates a synthetic internal training relationship.


In [ ]:
def resize_2d(image, new_hw, order=1):
    return resize(
        image,
        new_hw,
        order=order,
        mode='reflect',
        anti_aliasing=True,
        preserve_range=True
    ).astype(np.float32)

father = lr_slice
son_shape = (father.shape[0] // SCALE, father.shape[1] // SCALE)
son = resize_2d(father, son_shape, order=1)

print('Father shape:', father.shape)
print('Son shape:', son.shape)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(father.T, cmap='gray', origin='lower')
axes[0].set_title(f'Father / LR {father.shape}')
axes[1].imshow(son.T, cmap='gray', origin='lower')
axes[1].set_title(f'Son / LLR {son.shape}')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


## Module 7 — Bicubic interpolation baseline

Before training a CNN, establish a simple baseline. Bicubic interpolation upsamples the son back to the father size.

This is **not yet the final SR experiment**. It is a sanity check showing what ordinary interpolation does.


In [ ]:
bicubic_father = resize_2d(son, father.shape, order=3)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(father.T, cmap='gray', origin='lower')
axes[0].set_title('Father / reference for this internal experiment')
axes[1].imshow(son.T, cmap='gray', origin='lower')
axes[1].set_title('Son / input')
axes[2].imshow(bicubic_father.T, cmap='gray', origin='lower')
axes[2].set_title('Bicubic son → father')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()


## Module 8 — Extract self-supervised training patches

A single father/son pair can produce many local examples.

Each training example is:

`son patch → corresponding father patch`

For a ×2 experiment, a `32 × 32` son patch corresponds to a `64 × 64` father patch.

> In a more complete ZSSR implementation, one would also use multiple internal scales, random transformations, carefully designed degradation models, and other training details. Here we keep one scale for clarity.


In [ ]:
def extract_training_patches(son, father, patch_size=32, scale=2, stride=8, max_patches=4000):
    target_size = patch_size * scale
    xs, ys = [], []

    max_y = son.shape[0] - patch_size + 1
    max_x = son.shape[1] - patch_size + 1

    for y in range(0, max_y, stride):
        for x in range(0, max_x, stride):
            x_patch = son[y:y + patch_size, x:x + patch_size]
            y_patch = father[y * scale:y * scale + target_size,
                              x * scale:x * scale + target_size]
            if y_patch.shape == (target_size, target_size):
                xs.append(x_patch)
                ys.append(y_patch)

    if not xs:
        raise ValueError('No patches were generated. Reduce PATCH_SIZE or check image dimensions.')

    indices = np.arange(len(xs))
    np.random.shuffle(indices)
    indices = indices[:max_patches]

    X = np.stack([xs[i] for i in indices])[:, None, :, :]
    Y = np.stack([ys[i] for i in indices])[:, None, :, :]
    return X.astype(np.float32), Y.astype(np.float32)

X_np, Y_np = extract_training_patches(
    son,
    father,
    patch_size=PATCH_SIZE,
    scale=SCALE
)

print('Input patches:', X_np.shape)
print('Target patches:', Y_np.shape)


## Module 9 — Build a small ZSSR CNN

The network is intentionally simple. It first upsamples the son patch and then learns a nonlinear refinement.

This architecture is easier for students to understand than a large modern SR network.


In [ ]:
class SimpleZSSR(nn.Module):
    def __init__(self, scale=2):
        super().__init__()
        self.scale = scale
        self.refine = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 3, padding=1)
        )

    def forward(self, x):
        x = F.interpolate(x, scale_factor=self.scale, mode='bicubic', align_corners=False)
        residual = self.refine(x)
        return torch.clamp(x + residual, 0.0, 1.0)

model = SimpleZSSR(scale=SCALE).to(DEVICE)
print(model)


## Module 10 — Train the CNN

The only supervised-looking pair here is generated internally from the same low-field slice:

`son patches → father patches`

No external training dataset is required.


In [ ]:
X = torch.from_numpy(X_np)
Y = torch.from_numpy(Y_np)

loader = DataLoader(
    TensorDataset(X, Y),
    batch_size=BATCH_SIZE,
    shuffle=True
)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

loss_history = []

model.train()
for epoch in range(EPOCHS):
    running_loss = 0.0

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * xb.size(0)

    epoch_loss = running_loss / len(loader.dataset)
    loss_history.append(epoch_loss)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f'Epoch {epoch + 1:3d}/{EPOCHS} | Loss: {epoch_loss:.6f}')

plt.figure(figsize=(7, 4))
plt.plot(loss_history)
plt.xlabel('Epoch')
plt.ylabel('MSE loss')
plt.title('ZSSR training loss')
plt.grid(True, alpha=0.3)
plt.show()


## Module 11 — ZSSR inference

Now we perform the important step: feed the **original father/LR slice** to the CNN.

`father → ZSSR CNN → predicted SR`

The CNN has learned from the smaller son/father relationship and now applies that learned internal image prior to the father image.


In [ ]:
def zssr_inference(model, image, scale=2, device=DEVICE):
    model.eval()
    x = torch.from_numpy(image[None, None]).float().to(device)
    with torch.no_grad():
        sr = model(x).cpu().numpy()[0, 0]
    return np.clip(sr, 0, 1)

zssr_sr = zssr_inference(model, father, SCALE)

print('Input father shape:', father.shape)
print('ZSSR output shape:', zssr_sr.shape)


## Module 12 — Controlled evaluation against the 3T image

For a meaningful 3T-vs-64 mT comparison, the volumes should have compatible orientation, field of view, voxel spacing, and registration. We therefore do **not** automatically claim that the 3T image is a ground-truth target.

For the workshop demonstration, we first resize the selected 3T slice to the ZSSR output size so that students can see the mechanics of metric calculation. Treat these numbers as illustrative unless the two scans have been properly registered and harmonized.


In [ ]:
reference_3T_slice = mri_3T_n[:, :, SLICE_INDEX]
reference_3T_resized = resize_2d(reference_3T_slice, zssr_sr.shape, order=3)

bicubic_sr = resize_2d(father, zssr_sr.shape, order=3)

def calculate_metrics(reference, prediction):
    reference = np.clip(reference, 0, 1)
    prediction = np.clip(prediction, 0, 1)
    psnr = peak_signal_noise_ratio(reference, prediction, data_range=1.0)
    ssim = structural_similarity(reference, prediction, data_range=1.0)
    return psnr, ssim

psnr_bicubic, ssim_bicubic = calculate_metrics(reference_3T_resized, bicubic_sr)
psnr_zssr, ssim_zssr = calculate_metrics(reference_3T_resized, zssr_sr)

print('Illustrative metrics against resized 3T slice')
print(f'Bicubic: PSNR={psnr_bicubic:.3f} dB | SSIM={ssim_bicubic:.4f}')
print(f'ZSSR:    PSNR={psnr_zssr:.3f} dB | SSIM={ssim_zssr:.4f}')


## Module 13 — Visual comparison

Look for:

- edge preservation
- apparent sharpness
- noise amplification
- ringing or hallucinated structures
- anatomical consistency

For MRI, a visually sharper result is **not automatically a clinically better result**.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))

axes[0].imshow(reference_3T_resized.T, cmap='gray', origin='lower')
axes[0].set_title('3T reference*')

axes[1].imshow(bicubic_sr.T, cmap='gray', origin='lower')
axes[1].set_title(f'Bicubic\nPSNR {psnr_bicubic:.2f} | SSIM {ssim_bicubic:.3f}')

axes[2].imshow(zssr_sr.T, cmap='gray', origin='lower')
axes[2].set_title(f'ZSSR\nPSNR {psnr_zssr:.2f} | SSIM {ssim_zssr:.3f}')

error = np.abs(reference_3T_resized - zssr_sr)
axes[3].imshow(error.T, cmap='magma', origin='lower')
axes[3].set_title('Absolute error: ZSSR')

for ax in axes:
    ax.axis('off')

plt.suptitle('*Use as a quantitative reference only after registration/harmonization is established.')
plt.tight_layout()
plt.show()


## Module 14 — A fairer synthetic benchmark

A very useful teaching experiment is to create a known HR target from the 3T image itself.

Workflow:

`3T HR slice → synthetic downsample → synthetic LR → Bicubic/ZSSR → compare with original 3T HR`

This gives a clean ground-truth experiment because the HR reference is known. It also avoids interpreting differences between separate 3T and 64 mT acquisitions as pure SR error.


In [ ]:
# Synthetic benchmark using the 3T slice as known HR reference.
hr = mri_3T_n[:, :, SLICE_INDEX]

# Make dimensions divisible by SCALE for a simple demonstration.
h = (hr.shape[0] // SCALE) * SCALE
w = (hr.shape[1] // SCALE) * SCALE
hr = hr[:h, :w]

synthetic_lr = resize_2d(hr, (h // SCALE, w // SCALE), order=1)
synthetic_bicubic = resize_2d(synthetic_lr, hr.shape, order=3)

print('HR:', hr.shape)
print('Synthetic LR:', synthetic_lr.shape)

psnr_syn_bicubic, ssim_syn_bicubic = calculate_metrics(hr, synthetic_bicubic)
print(f'Synthetic bicubic baseline: PSNR={psnr_syn_bicubic:.3f} dB | SSIM={ssim_syn_bicubic:.4f}')


## Module 15 — From one slice to the complete volume

Once the single-slice pipeline works, we can apply the same process independently to each axial slice.

Conceptually:

`Volume → slice 1 → ZSSR → SR slice 1`

`       → slice 2 → ZSSR → SR slice 2`

`       → ...`

`       → slice N → ZSSR → SR slice N`

The outputs are then stacked back into a 3D volume.

> **Important:** Independent 2D processing does not learn through-plane information. It can also produce slice-to-slice inconsistency. This is why the next notebook should investigate 2.5D and full 3D approaches.


## Module 16 — ZSSR improvement experiments

Students can now change one variable at a time and record PSNR/SSIM.

| Experiment | Change |
|---|---|
| Baseline | Simple CNN + MSE |
| 1 | L1 loss |
| 2 | More CNN layers |
| 3 | Different patch size |
| 4 | More training patches |
| 5 | More epochs |
| 6 | Learning-rate tuning |
| 7 | Residual vs non-residual CNN |
| 8 | Different degradation kernel |
| 9 | Data augmentation |

Do not assume an architectural change improves performance. The experiment should demonstrate whether it does.


# Module 17 — Extension to 3D ZSSR

The 2D idea generalizes naturally to a volume:

`HR volume → LR volume → lower-resolution volume`

and then:

`lower-resolution volume → 3D CNN → LR volume`

followed by:

`LR volume → trained 3D CNN → SR volume`

### Three useful paths

1. **Slice-wise 2D ZSSR:** easiest and uses the current notebook almost unchanged.
2. **2.5D ZSSR:** use neighboring slices as input channels and predict the center slice.
3. **Full 3D ZSSR:** use 3D patches and `Conv3d` layers; this is the most direct volumetric extension but requires substantially more GPU memory and careful handling of voxel spacing.

For LF-MRI, start with slice-wise 2D, then 2.5D, and only then move to full 3D.


# Module 18 — Key takeaways

By the end of this notebook, students should be able to explain:

- why bicubic interpolation is a useful baseline;
- what makes ZSSR different from ordinary supervised SR;
- the father/son internal training relationship;
- how patches are generated from one image;
- how a small CNN learns the internal scale relationship;
- why PSNR/SSIM require a valid reference;
- why real LF-MRI requires additional validation beyond image sharpness;
- why independent 2D slice processing is only the first step toward 3D SR.

### Final conceptual diagram

```text
64 mT MRI slice (Father / LR)
          │
          ├── downsample ×2 ──> Son / LLR
          │                         │
          │                         ▼
          │                    train CNN
          │                         │
          └─────────────────────────┘
                                    │
                                    ▼
                         CNN(Father / LR)
                                    │
                                    ▼
                              ZSSR / SR
```

The next notebook can turn this into a complete **volume-level ZSSR pipeline**, including slice-by-slice inference, NIfTI output, 3T/64 mT registration-aware evaluation, and experiments with 2.5D and 3D SR.
